In [1]:
import tkinter as tk
from tkinter import Canvas, simpledialog
import random

In [2]:
# Color Constants
BACKGROUND_COLOR = "white"
TEXT_COLOR = "black"
START_COLOR = "green"
END_COLOR = "red"
ACTIVE_COLOR = "blue"
INACTIVE_COLOR = "lightgray"
BUTTON_BG = "lightgray"
BUTTON_FG = "black"

In [3]:
class CanvasValues:
    WIDTH, HEIGHT = 400, 500
    START, STEP, END = 100, 100, 300
    RADIUS = 30


In [4]:
class DrawingCanvas:
    def __init__(self, root):
        self.canvas = Canvas(root, width=CanvasValues.WIDTH, height=CanvasValues.HEIGHT - 100, bg=BACKGROUND_COLOR)
        self.canvas.pack(fill=tk.BOTH, expand=True)
    
    def draw_line(self, x1, y1, x2, y2):
        self.canvas.create_line(x1, y1, x2, y2, fill=ACTIVE_COLOR, width=4)
    
    def draw_circle(self, x, y, r, outline=INACTIVE_COLOR, fill_color=BACKGROUND_COLOR, number=None):
        self.canvas.create_oval(x - r, y - r, x + r, y + r, outline=outline, fill=fill_color, width=3)
        if number is not None:
            self.canvas.create_text(x, y, text=str(number), font=("Arial", 16), fill=TEXT_COLOR)
    
    def clear(self):
        self.canvas.delete("all")


In [5]:
class PatternGenerator:
    
    VALID_MOVES = {
        1: {2, 4, 5, 6, 8}, 
        2: {1, 3, 4, 5, 6, 7, 9}, 
        3: {2, 5, 6, 8, 9},
        4: {1, 2, 5, 7, 8}, 
        5: {1, 2, 3, 4, 6, 7, 8, 9}, 
        6: {2, 3, 5, 8, 9},
        7: {4, 5, 8}, 
        8: {4, 5, 6, 7, 9}, 
        9: {5, 6, 8}
    }
    
    CROSSOVERS = {
        (1, 3): 2, 
        (3, 1): 2, 
        (1, 7): 4, 
        (7, 1): 4, 
        (1, 9): 5,
        (9, 1): 5, 
        
        (2, 8): 5, 
        (8, 2): 5, 
        
        (3, 7): 5, 
        (7, 3): 5,
        (3, 9): 6, 
        (9, 3): 6,
        
        (4, 6): 5, 
        (6, 4): 5, 
        
        (7, 9): 8, 
        (9, 7): 8
    }

    @staticmethod
    def get_valid_pattern(length):
        while True:
            start = random.randint(1, 9)
            sequence, used = [start], {start}
            while len(sequence) < length:
                next_moves = [p for p in PatternGenerator.VALID_MOVES[sequence[-1]] if p not in used]
                next_moves = [p for p in next_moves if p not in PatternGenerator.CROSSOVERS or PatternGenerator.CROSSOVERS[p, sequence[-1]] in used]
                if not next_moves:
                    break
                next_point = random.choice(next_moves)
                sequence.append(next_point)
                used.add(next_point)
            if len(sequence) == length:
                return sequence



In [6]:
class LockScreenApp:
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("Android Lock Screen")
        self.root.configure(bg=BACKGROUND_COLOR)
        self.canvas = DrawingCanvas(self.root)
        self.sequence_label = tk.Label(self.root, text="", fg=TEXT_COLOR, bg=BACKGROUND_COLOR, font=("Arial", 14))
        self.sequence_label.pack(pady=5)
        self.length_var = tk.IntVar(value=4)
        self.create_controls()
        self.generate_pattern()
    
    def create_controls(self):
        control_frame = tk.Frame(self.root, bg=BACKGROUND_COLOR)
        control_frame.pack(pady=10)
        
        tk.Label(control_frame, text="Pattern Length:", fg=TEXT_COLOR, bg=BACKGROUND_COLOR, font=("Arial", 12)).pack(side=tk.LEFT)
        tk.Scale(control_frame, from_=4, to=9, orient=tk.HORIZONTAL, variable=self.length_var, length=150, bg=BACKGROUND_COLOR, fg=TEXT_COLOR).pack(side=tk.LEFT, padx=10)
        tk.Button(control_frame, text="Generate Pattern", command=self.generate_pattern, bg=BUTTON_BG, fg=BUTTON_FG).pack(side=tk.LEFT)
    
    def generate_pattern(self):
        sequence = PatternGenerator.get_valid_pattern(self.length_var.get())
        self.draw_pattern(sequence)
    
    def draw_pattern(self, sequence):
        coords = {num: ((num - 1) % 3 * 100 + 100, (num - 1) // 3 * 100 + 100) for num in range(1, 10)}
        self.canvas.clear()
        
        for num in range(1, 10):
            x, y = coords[num]
            if num == sequence[0]:
                color = START_COLOR  
            elif num == sequence[-1]:
                color = END_COLOR  
            elif num in sequence:
                color = ACTIVE_COLOR
            else:
                color = INACTIVE_COLOR  
            self.canvas.draw_circle(x, y, CanvasValues.RADIUS, outline=color, fill_color=BACKGROUND_COLOR, number=num)
        
        for i in range(1, len(sequence)):
            self.canvas.draw_line(*coords[sequence[i-1]], *coords[sequence[i]])
        
        self.sequence_label.config(text=f"Pattern Sequence: {'-'.join(map(str, sequence))}")
    
    def run(self):
        self.root.mainloop()




In [10]:
if __name__ == "__main__":
    LockScreenApp().run()